# Week3 Assignment: Sequence Prediction with LSTM Networks
## Encoder-Decoder LSTMs for Learning Mathematical Operations

---

### Assignment Overview

In this assignment, you will build two LSTM-based models from scratch:

1. **Part A — Addition as a Mapping Problem:** Train a simple LSTM to directly map pairs of numbers to their sum (the "beginner's approach").
2. **Part B — Addition as a Seq2Seq Problem:** Frame addition as a true *sequence-to-sequence* prediction task, treating the arithmetic expression `"12+50"` as a character sequence and the result `"62"` as the output sequence. This is the correct, powerful use of LSTMs.
3. **Part C — Extensions & Analysis:** Extend the model and reflect critically on the differences.

### Learning Objectives

By completing this assignment, you will:
- Understand the difference between a **mapping problem** and a **sequence prediction problem**
- Implement a **stacked LSTM** for regression
- Implement an **Encoder-Decoder (Seq2Seq) LSTM** architecture with `RepeatVector` and `TimeDistributed` layers
- Understand **one-hot encoding** for character-level sequence modelling
- Evaluate and compare model performance using RMSE and accuracy

### Submission Instructions

- Run **all cells** before submitting (Kernel → Restart & Run All)
- Answer all written questions in the provided Markdown cells
- Do **not** delete any `# TODO` comments — fill them in
- Submit as `.ipynb`

---

---
## Section 0: Environment Setup

Run the cell below to install and import all required libraries. This assignment uses **TensorFlow/Keras**, NumPy, and scikit-learn.

> 💡 If you are on Google Colab, TensorFlow is pre-installed. On a local machine, run `pip install tensorflow scikit-learn` if needed.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from numpy import array, argmax
from random import seed, randint
from math import ceil, log10, sqrt
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, TimeDistributed, RepeatVector

# Fix random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print("All imports successful")

---
## Part A: Addition as a Mapping Problem

### Background

A common beginner mistake when learning LSTMs is to frame a problem that *looks* sequential as a plain **mapping (function approximation)** task. When adding two numbers `[a, b] → a+b`, the order of `a` and `b` does not matter — you could shuffle the inputs and still get the right answer. This is **not** a sequence problem.

In this part, you will:
1. Generate training data (pairs of random integers and their sum)
2. Normalize the data
3. Build a simple stacked LSTM for regression
4. Evaluate with RMSE

> **Key insight:** After completing Part A and B, reflect on *why* the MLP and simple LSTM work just as well (or better) than a seq2seq model on this framing.

### Task A1: Data Generation

**Direction:** Implement the `random_sum_pairs` function below. It should:
- Generate `n_examples` pairs of random integers, each in the range `[1, largest]`
- Compute the sum of each pair
- Return normalized `X` and `y` arrays (divide by `largest * n_numbers`)

The normalization maps all values into `[0, 1]`, which is required for the LSTM's activation functions.

In [ ]:
def random_sum_pairs(n_examples, n_numbers, largest):
    """
    Generate random integer pairs and their normalized sum.

    Parameters
    ----------
    n_examples : int  – number of samples to generate
    n_numbers  : int  – how many integers to add per sample
    largest    : int  – maximum value of each integer (inclusive)

    Returns
    -------
    X : np.ndarray of shape (n_examples, n_numbers) – normalized inputs
    y : np.ndarray of shape (n_examples,)            – normalized sums
    """
    # TODO ────────────────────────────────────────────────────────────────────
    # 1. Create empty lists X_list and y_list
    # 2. Loop n_examples times:
    #      a. Generate a list of n_numbers random integers between 1 and largest
    #      b. Compute the sum of those integers
    #      c. Append the list to X_list and the sum to y_list
    # 3. Convert X_list and y_list to numpy arrays
    # 4. Normalize: divide X by (largest * n_numbers), y by (largest * n_numbers)
    # 5. Return X, y
    # ─────────────────────────────────────────────────────────────────────────
    X_list, y_list = [], []
    for _ in range(n_examples):
        inp = [randint(1, largest) for _ in range(n_numbers)]
        X_list.append(inp)
        y_list.append(sum(inp))
    X = np.array(X_list)
    y = np.array(y_list)
    X = X / (largest * n_numbers)
    y = y / (largest * n_numbers)
    return X, y


def invert_sum(value, n_numbers, largest):
    """Convert a normalized prediction back to the original integer scale."""
    # TODO: multiply value by (largest * n_numbers) and round to nearest int
    return round(value * (largest * n_numbers))

# ── Quick sanity check ───────────────────────────────────────────────────────
seed(1)
X_test, y_test = random_sum_pairs(5, 2, 10)
print("Sample X (normalized):", X_test[:3])
print("Sample y (normalized):", y_test[:3])
print("Reconstructed sums   :", [invert_sum(v, 2, 10) for v in y_test[:3]])

### Task A2: Build the LSTM Model

**Direction:** Build a stacked LSTM that accepts a sequence of `n_numbers` time steps (each with 1 feature) and outputs a single scalar (the predicted normalized sum).

Architecture to implement:
- `LSTM(10, input_shape=(n_numbers, 1))` — one hidden LSTM layer with 10 units
- `Dense(1)` — output layer (regression)
- Compile with **`mean_squared_error`** loss and **`adam`** optimizer

> The input must be reshaped to `(n_examples, n_numbers, 1)` — a 3D tensor of shape (samples, timesteps, features).

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
seed(1)
n_examples = 100
n_numbers  = 2
largest    = 100
n_batch    = 1
n_epoch    = 100

# ── TODO: Build the model ────────────────────────────────────────────────────

model_a = Sequential([
    LSTM(10, input_shape=(n_numbers, 1)),
    Dense(1)
])
model_a.compile(loss='mean_squared_error', optimizer='adam')

# TODO: print the model summary
model_a.summary()

### Task A3: Train the Model

**Direction:** Train `model_a` for `n_epoch` epochs. On each epoch:
1. Generate a fresh batch of `n_examples` samples using `random_sum_pairs`
2. Reshape `X` to `(n_examples, n_numbers, 1)` (add feature dimension)
3. Call `model_a.fit(X, y, epochs=1, batch_size=n_batch, verbose=0)`

Store the loss from each epoch in a list called `history_a` for plotting.

> Generating new data each epoch is a form of **online data augmentation** — the model never sees the exact same batch twice, which helps it generalize.

In [ ]:
history_a = []

# TODO ────────────────────────────────────────────────────────────────────────
for epoch in range(n_epoch):
    X, y = random_sum_pairs(n_examples, n_numbers, largest)
    X = X.reshape((n_examples, n_numbers, 1))
    h = model_a.fit(X, y, epochs=1, batch_size=n_batch, verbose=0)
    history_a.append(h.history['loss'][0])
# ─────────────────────────────────────────────────────────────────────────────

# ── Plot training loss ───────────────────────────────────────────────────────
# TODO: plot history_a with plt.plot — label axes and add a title
plt.plot(history_a)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Part A — LSTM Training Loss')
plt.show()

### Task A4: Evaluate the Model

**Direction:** Evaluate `model_a` on **100 new, unseen** samples:
1. Generate test data with `random_sum_pairs`
2. Reshape `X` to `(n_examples, n_numbers, 1)`
3. Predict with `model_a.predict`
4. Invert the normalization using `invert_sum`
5. Compute **RMSE** using `mean_squared_error` from sklearn
6. Print the first 20 predictions alongside the expected values and error

>  **Reflection Question (written answer below):** Is RMSE a good metric here? What metric would you consider "acceptable" for this problem?

In [ ]:
# TODO ────────────────────────────────────────────────────────────────────────
X_test, y_test = random_sum_pairs(100, n_numbers, largest)
X_test = X_test.reshape((100, n_numbers, 1))
y_pred = model_a.predict(X_test, verbose=0)

expected  = [invert_sum(v, n_numbers, largest) for v in y_test]
predicted = [invert_sum(v[0], n_numbers, largest) for v in y_pred]

rmse = sqrt(mean_squared_error(expected, predicted))
print(f"RMSE: {rmse:.2f}")
print(f"\n{'Expected':>10} {'Predicted':>10} {'Error':>10}")
for i in range(20):
    print(f"{expected[i]:>10} {predicted[i]:>10} {abs(expected[i]-predicted[i]):>10}")
# ─────────────────────────────────────────────────────────────────────────────

#### Written Answer – Part A

**Q1.** What RMSE did you achieve? Is the model learning the task correctly?

*The model achieves low RMSE (typically 1–5), meaning it learns the sum correctly.*<br>
*Yes, the model is learning the task.*

---

**Q2.** Could a simple MLP (fully connected network) solve this same problem? Why or why not?

*Yes, a simple MLP can solve this just as well because addition is not a sequential problem — order of inputs doesn't matter. LSTMs add unnecessary complexity here.*

---

---
## Part B: Addition as a True Seq2Seq Problem

### Background

The "correct" and more powerful LSTM use-case treats the entire arithmetic expression as a **character-level sequence**. Given the input string `"12+50"` (character by character), the model must output the string `"62"` (character by character).

This is a **sequence-to-sequence (seq2seq)** problem:
- Input sequence: `['1', '2', '+', '5', '0']` → 5 timesteps, each one-hot encoded
- Output sequence: `['6', '2']` → 2 timesteps, each one-hot encoded

The architecture uses an **Encoder–Decoder** pattern with a `RepeatVector` bridge:

```
Input Chars  →  [Encoder LSTM]  →  Context Vector  →  [RepeatVector]
             →  [Decoder LSTM]  →  [TimeDistributed Dense]  →  Output Chars
```

In this part you will implement the full data pipeline and the model.

### Task B1: Convert Numbers to Padded Strings

**Direction:** Implement `to_string(X, y, n_numbers, largest)` that:
- Joins each input list like `[3, 10]` into the string `"3+10"`, padded on the left with spaces to fixed length
- Converts each sum integer to a right-padded string of fixed length
- Input max length formula: `n_numbers * ceil(log10(largest+1)) + n_numbers - 1`
- Output max length formula: `ceil(log10(n_numbers * (largest+1)))`

> **Why padding?** LSTMs require fixed-length input sequences. Padding with spaces allows variable-length numbers to be handled uniformly. The model learns to ignore the padding character.

In [ ]:
def to_string(X, y, n_numbers, largest):
    """
    Convert integer lists and sum values to fixed-length padded strings.

    Parameters
    ----------
    X         : list of lists – e.g. [[3, 10], [5, 7]]
    y         : list of ints  – e.g. [13, 12]
    n_numbers : int
    largest   : int

    Returns
    -------
    X_str : list of str – e.g. [' 3+10', ' 5+ 7']
    y_str : list of str – e.g. ['13', '12']
    """
    # TODO ────────────────────────────────────────────────────────────────────
    # Compute max input length:
    #   max_in = n_numbers * ceil(log10(largest + 1)) + n_numbers - 1
    # For each pattern in X:
    #   strp = '+'.join([str(n) for n in pattern])
    #   left-pad strp with spaces to reach max_in length
    # Compute max output length:
    #   max_out = ceil(log10(n_numbers * (largest + 1)))
    # For each value in y:
    #   strp = str(value)
    #   left-pad strp with spaces to reach max_out length
    # ─────────────────────────────────────────────────────────────────────────
    max_in  = n_numbers * ceil(log10(largest + 1)) + n_numbers - 1
    max_out = ceil(log10(n_numbers * (largest + 1)))
    X_str, y_str = [], []
    for pattern in X:
        strp = '+'.join([str(n) for n in pattern])
        strp = strp.rjust(max_in)
        X_str.append(strp)
    for value in y:
        strp = str(value).rjust(max_out)
        y_str.append(strp)
    return X_str, y_str

# ── Sanity check ─────────────────────────────────────────────────────────────
seed(1)
X_raw, y_raw = random_sum_pairs(3, 2, 10)
# NOTE: random_sum_pairs returns normalized arrays — for seq2seq we need raw ints
# We'll re-generate without normalization using a helper below
def raw_sum_pairs(n_examples, n_numbers, largest):
    X_list, y_list = [], []
    for _ in range(n_examples):
        inp = [randint(1, largest) for _ in range(n_numbers)]
        X_list.append(inp); y_list.append(sum(inp))
    return X_list, y_list

X_raw, y_raw = raw_sum_pairs(3, 2, 10)
X_str, y_str = to_string(X_raw, y_raw, 2, 10)
print("Raw inputs :", X_raw)
print("Str inputs :", X_str)
print("Str outputs:", y_str)

### Task B2: Integer Encode the Character Strings

**Direction:** Implement `integer_encode(X, y, alphabet)` that maps each character in every string to its index in `alphabet`.

The alphabet for this problem is:
```python
alphabet = ['0','1','2','3','4','5','6','7','8','9','+', ' ']
```
So `'0'` → `0`, `'+'` → `10`, `' '` (space) → `11`.

> This is a necessary stepping stone before one-hot encoding. It converts strings into sequences of integers that can be further encoded.

In [ ]:
ALPHABET = ['0','1','2','3','4','5','6','7','8','9','+', ' ']

def integer_encode(X, y, alphabet):
    """
    Map each character in each string to its index in alphabet.

    Returns
    -------
    X_enc : list of list of int
    y_enc : list of list of int
    """
    # TODO ────────────────────────────────────────────────────────────────────
    # char_to_int = {c: i for i, c in enumerate(alphabet)}
    # For each string pattern in X, convert each character to its integer index
    # Do the same for y
    # ─────────────────────────────────────────────────────────────────────────
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    X_enc = [[char_to_int[c] for c in pattern] for pattern in X]
    y_enc = [[char_to_int[c] for c in pattern] for pattern in y]
    return X_enc, y_enc



# ── Sanity check ─────────────────────────────────────────────────────────────
X_int, y_int = integer_encode(X_str, y_str, ALPHABET)
print("Integer-encoded X:", X_int)
print("Integer-encoded y:", y_int)

### Task B3: One-Hot Encode the Integer Sequences

**Direction:** Implement `one_hot_encode(X, y, n_classes)` that converts each integer index into a binary vector of length `n_classes` with a `1` at the index position and `0` elsewhere.

For example, index `3` with `n_classes=12` becomes:
```
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
```

> One-hot encoding frames each character position as a **multi-class classification** output. The model will use a softmax output layer and categorical cross-entropy loss.

In [ ]:
def one_hot_encode(X, y, n_classes):
    """
    Convert integer-encoded sequences to one-hot binary vectors.

    Returns
    -------
    X_enc : list of list of list (shape: [n_samples, seq_len, n_classes])
    y_enc : list of list of list (shape: [n_samples, out_len, n_classes])
    """
    # TODO ────────────────────────────────────────────────────────────────────
    # For each sample in X (a list of integer indices):
    #   For each index, create a zero vector of length n_classes
    #   Set vector[index] = 1
    # Do the same for y
    # ─────────────────────────────────────────────────────────────────────────
    def encode_seq(seq):
        result = []
        for idx in seq:
            vec = [0] * n_classes
            vec[idx] = 1
            result.append(vec)
        return result
    X_enc = [encode_seq(seq) for seq in X]
    y_enc = [encode_seq(seq) for seq in y]
    return X_enc, y_enc


# ── Sanity check ─────────────────────────────────────────────────────────────
X_ohe, y_ohe = one_hot_encode(X_int, y_int, len(ALPHABET))
print("Shape of X_ohe[0]:", array(X_ohe[0]).shape,
      "(should be (input_seq_len, 12))")
print("Shape of y_ohe[0]:", array(y_ohe[0]).shape,
      "(should be (output_seq_len, 12))")

### Task B4: Assemble the Full Data Pipeline

**Direction:** Implement `generate_data(n_samples, n_numbers, largest, alphabet)` that chains all steps:
1. `raw_sum_pairs` → integer lists
2. `to_string` → padded strings
3. `integer_encode` → integer sequences
4. `one_hot_encode` → binary tensors
5. Return as numpy arrays

In [ ]:
def generate_data(n_samples, n_numbers, largest, alphabet):
    """Full pipeline: integers → one-hot encoded numpy arrays."""
    # TODO ────────────────────────────────────────────────────────────────────
    # Call each function in order, then convert to numpy arrays
    # ─────────────────────────────────────────────────────────────────────────
    X_raw, y_raw = raw_sum_pairs(n_samples, n_numbers, largest)
    X_str, y_str = to_string(X_raw, y_raw, n_numbers, largest)
    X_int, y_int = integer_encode(X_str, y_str, alphabet)
    X_ohe, y_ohe = one_hot_encode(X_int, y_int, len(alphabet))
    return np.array(X_ohe), np.array(y_ohe)


def invert_ohe(seq, alphabet):
    """Decode a one-hot output sequence back to a string."""
    int_to_char = {i: c for i, c in enumerate(alphabet)}
    return ''.join(int_to_char[argmax(v)] for v in seq)


# ── Compute sequence length constants ────────────────────────────────────────
seed(1)
n_samples_b  = 1000
n_numbers_b  = 2
largest_b    = 10
n_chars      = len(ALPHABET)
n_in_len     = n_numbers_b * ceil(log10(largest_b + 1)) + n_numbers_b - 1
n_out_len    = ceil(log10(n_numbers_b * (largest_b + 1)))

print(f"Input  sequence length : {n_in_len}")
print(f"Output sequence length : {n_out_len}")
print(f"Alphabet size          : {n_chars}")

# ── Test generate_data ────────────────────────────────────────────────────────
X_b, y_b = generate_data(4, n_numbers_b, largest_b, ALPHABET)
print(f"\nX shape: {X_b.shape}  (should be (4, {n_in_len}, {n_chars}))")
print(f"y shape: {y_b.shape}  (should be (4, {n_out_len}, {n_chars}))")

### Task B5: Build the Encoder-Decoder LSTM Model

**Direction:** Build the seq2seq LSTM model with the following architecture:

```
Input: (n_in_len, n_chars)
  └─ Encoder LSTM (100 units)  → Context vector (100,)
       └─ RepeatVector(n_out_len) → (n_out_len, 100)
            └─ Decoder LSTM (50 units, return_sequences=True)
                 └─ TimeDistributed(Dense(n_chars, activation='softmax'))
Output: (n_out_len, n_chars)
```

- **Encoder:** A single LSTM that reads the input sequence and compresses it to a fixed-size context vector
- **RepeatVector:** Repeats the context vector `n_out_len` times so the decoder can produce a sequence
- **Decoder:** An LSTM + Dense that generates one output character at a time
- **Loss:** `categorical_crossentropy` | **Optimizer:** `adam` | **Metric:** `accuracy`

> **Why RepeatVector?** The encoder outputs a 2D tensor `(batch, 100)`. The decoder LSTM needs a 3D input `(batch, timesteps, features)`. `RepeatVector(n)` repeats the vector `n` times, bridging the gap.

In [ ]:
# ── Define hyperparameters ────────────────────────────────────────────────────
n_batch_b  = 10
n_epoch_b  = 30

# TODO: Build the Encoder-Decoder LSTM ────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────

model_b = Sequential([
    LSTM(100, input_shape=(n_in_len, n_chars)),
    RepeatVector(n_out_len),
    LSTM(50, return_sequences=True),
    TimeDistributed(Dense(n_chars, activation='softmax'))
])
model_b.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_b.summary()

### Task B6: Train the Seq2Seq Model

**Direction:** Train `model_b` for `n_epoch_b` epochs. Each epoch:
1. Generate `n_samples_b` fresh examples with `generate_data`
2. Fit for 1 epoch with `batch_size=n_batch_b`
3. Record both `loss` and `accuracy` for plotting

In [ ]:
history_b_loss = []
history_b_acc  = []

# TODO ────────────────────────────────────────────────────────────────────────
for epoch in range(n_epoch_b):
    X_b, y_b = generate_data(n_samples_b, n_numbers_b, largest_b, ALPHABET)
    h = model_b.fit(X_b, y_b, epochs=1, batch_size=n_batch_b, verbose=0)
    history_b_loss.append(h.history['loss'][0])
    history_b_acc.append(h.history['accuracy'][0])
# ─────────────────────────────────────────────────────────────────────────────

# ── Plot ──────────────────────────────────────────────────────────────────────
# TODO: Plot both loss and accuracy curves on side-by-side subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history_b_loss); ax1.set_title('Part B Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax2.plot(history_b_acc);  ax2.set_title('Part B Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
plt.tight_layout(); plt.show()

### Task B7: Evaluate the Seq2Seq Model

**Direction:** Generate 100 test samples and evaluate the model:
1. Predict with `model_b.predict`
2. Decode predictions using `invert_ohe`
3. Print the first 20 (expected, predicted) pairs
4. Compute **character-level accuracy**: the percentage of predictions that exactly match the expected string

> An "exact match" means the model predicted *every character correctly* — a stricter metric than element-wise accuracy.

In [ ]:
# TODO ────────────────────────────────────────────────────────────────────────
X_test_b, y_test_b = generate_data(100, n_numbers_b, largest_b, ALPHABET)
y_pred_b = model_b.predict(X_test_b, verbose=0)

correct = 0
print(f"{'Expected':>6} {'Predicted':>10}")
for i in range(100):
    expected_str  = invert_ohe(y_test_b[i], ALPHABET).strip()
    predicted_str = invert_ohe(y_pred_b[i], ALPHABET).strip()
    if i < 20:
        print(f"{expected_str:>6} {predicted_str:>10}")
    if expected_str == predicted_str:
        correct += 1

print(f"\nExact-match accuracy: {correct}%")
# ─────────────────────────────────────────────────────────────────────────────

#### Written Answer – Part B

**Q3.** What exact-match accuracy did your seq2seq model achieve? Does increasing `n_epoch_b` improve results?

*Accuracy typically reaches 80–95% in 30 epochs. Increasing n_epoch_b improves it.*

---

**Q4.** Explain the role of `RepeatVector` in the encoder-decoder architecture. What would happen if you removed it and directly connected the encoder output to the decoder?

*RepeatVector repeats the encoder's context vector n_out_len times, converting shape (batch, 100) → (batch, n_out_len, 100) so the decoder LSTM gets a proper 3D input. Without it, the decoder can't produce a sequence output.*

---

**Q5.** Why is `TimeDistributed(Dense(...))` used instead of a plain `Dense` layer for the output?

*TimeDistributed applies the same Dense layer independently to each time step. A plain Dense would only work on the last time step, not produce a sequence output.*

---

---
## Part C : Extensions & Critical Analysis

This section challenges you to go beyond the baseline and critically compare the two approaches.

> These tasks are designed to deepen your understanding. Partial credit is given for thoughtful reasoning even if results are not perfect.

### Task C1: MLP Baseline Comparison

**Direction:** Build a simple **Multi-Layer Perceptron (MLP)** to solve the same mapping problem as Part A. Compare its RMSE to the LSTM's RMSE.

Architecture:
- `Dense(4, input_dim=n_numbers, activation='relu')`
- `Dense(2, activation='relu')`
- `Dense(1)`

Train for 50 epochs, batch_size=2, same data generation as Part A.

In [ ]:
# TODO ────────────────────────────────────────────────────────────────────────
# Build, train, and evaluate the MLP.
# Print RMSE and compare to Part A.
# ─────────────────────────────────────────────────────────────────────────────
model_mlp = Sequential([
    Dense(4, input_dim=n_numbers, activation='relu'),
    Dense(2, activation='relu'),
    Dense(1)
])
model_mlp.compile(loss='mean_squared_error', optimizer='adam')

for epoch in range(50):
    X, y = random_sum_pairs(n_examples, n_numbers, largest)
    model_mlp.fit(X, y, epochs=1, batch_size=2, verbose=0)

X_test, y_test = random_sum_pairs(100, n_numbers, largest)
y_pred_mlp = model_mlp.predict(X_test, verbose=0)
expected_mlp  = [invert_sum(v, n_numbers, largest) for v in y_test]
predicted_mlp = [invert_sum(v[0], n_numbers, largest) for v in y_pred_mlp]
rmse_mlp = sqrt(mean_squared_error(expected_mlp, predicted_mlp))
print(f"MLP RMSE:  {rmse_mlp:.2f}")
print(f"LSTM RMSE: {rmse:.2f}")
# ─────────────────────────────────────────────────────────────────────────────

#### Written Answer – Task C1

**Q6.** Which model (MLP vs LSTM) achieved lower RMSE for the mapping problem? What does this reveal about when LSTMs are the right tool?

*MLP typically achieves similar or better RMSE than LSTM for this task. This shows LSTMs are overkill for simple mapping problems where order doesn't matter. LSTMs shine when temporal dependencies across time steps are important.*

---

### Task C2: Scale to Larger Numbers

**Direction:** Retrain your seq2seq model (`model_b`) but change the dataset configuration to **3 numbers in the range 1–99** (i.e., `n_numbers=3, largest=99`). You will need to:
1. Recompute `n_in_len` and `n_out_len`
2. Rebuild and retrain `model_b` with the new input/output shapes
3. Report the final accuracy

> You may need to increase `n_epoch_b` to 50 or more. Note how the model complexity needs to scale with the input difficulty.

In [ ]:
# TODO ────────────────────────────────────────────────────────────────────────
# n_numbers_c = 3
# largest_c   = 99
# Recompute n_in_len_c, n_out_len_c
# Rebuild and retrain the seq2seq model
# Evaluate and report accuracy
# ─────────────────────────────────────────────────────────────────────────────
n_numbers_c = 3
largest_c   = 99
n_in_len_c  = n_numbers_c * ceil(log10(largest_c + 1)) + n_numbers_c - 1
n_out_len_c = ceil(log10(n_numbers_c * (largest_c + 1)))

model_c = Sequential([
    LSTM(100, input_shape=(n_in_len_c, n_chars)),
    RepeatVector(n_out_len_c),
    LSTM(50, return_sequences=True),
    TimeDistributed(Dense(n_chars, activation='softmax'))
])
model_c.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

for epoch in range(50):
    X_c, y_c = generate_data(n_samples_b, n_numbers_c, largest_c, ALPHABET)
    model_c.fit(X_c, y_c, epochs=1, batch_size=n_batch_b, verbose=0)

X_test_c, y_test_c = generate_data(100, n_numbers_c, largest_c, ALPHABET)
y_pred_c = model_c.predict(X_test_c, verbose=0)
correct_c = sum(
    invert_ohe(y_test_c[i], ALPHABET).strip() == invert_ohe(y_pred_c[i], ALPHABET).strip()
    for i in range(100)
)
print(f"Scaled model accuracy: {correct_c}%")
# ─────────────────────────────────────────────────────────────────────────────

#### Written Answer – Task C2

**Q7.** How did accuracy change when you scaled from 2 numbers (1–10) to 3 numbers (1–99)? What architectural or training changes helped?

*Accuracy dropped noticeably when scaling from 2 numbers (1–10) to 3 numbers (1–99) because the task became more complex and required handling longer sequences and more carry operations. Increasing model capacity, training data, and training epochs can help improve performance on the larger task.*

---

### Task C3: Loss Curve Analysis & Visualization

**Direction:** Plot a combined figure with:
1. Part A LSTM loss curve
2. Part B seq2seq loss and accuracy curves

Then write a brief analysis below comparing the convergence behaviour.

In [ ]:
# TODO ────────────────────────────────────────────────────────────────────────
# Create a figure with 3 subplots side by side:
#   1. Part A loss (history_a)
#   2. Part B loss (history_b_loss)
#   3. Part B accuracy (history_b_acc)
# Label all axes. Add a title to each subplot.
# ─────────────────────────────────────────────────────────────────────────────
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.plot(history_a);        ax1.set_title('Part A LSTM Loss');    ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss')
ax2.plot(history_b_loss);   ax2.set_title('Part B Seq2Seq Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('CE Loss')
ax3.plot(history_b_acc);    ax3.set_title('Part B Accuracy');     ax3.set_xlabel('Epoch'); ax3.set_ylabel('Accuracy')
plt.tight_layout(); plt.show()


#### Written Answer – Task C3

**Q8.** Describe the shape of the loss curves. Did either model show signs of overfitting? Which converged faster?

*Part A loss drops quickly and smoothly — simple task. Part B loss drops more slowly as the model learns character-level patterns. No significant overfitting since new data is generated each epoch. Part B converges slower but reaches higher capability.*

---

### Task C4: Error Analysis – Where Does the Seq2Seq Model Fail?

**Direction:** Generate 500 test samples and identify:
1. Which predicted strings were **incorrect** (expected ≠ predicted)
2. Whether errors tend to occur on certain **types of inputs** (e.g., sums with carry digits, larger numbers, etc.)
3. Print 10 representative failure cases

> This is an open-ended analysis task. There's no single correct answer — the goal is to reason carefully about model limitations.

In [ ]:
# TODO ────────────────────────────────────────────────────────────────────────
# Generate 500 test samples
# Identify all incorrect predictions
# Analyze patterns in failures
# Print 10 failure cases with their inputs (re-generate the raw inputs alongside)
# ─────────────────────────────────────────────────────────────────────────────
X_err, y_err = generate_data(500, n_numbers_b, largest_b, ALPHABET)
y_pred_err = model_b.predict(X_err, verbose=0)

failures = []
for i in range(500):
    exp  = invert_ohe(y_err[i],      ALPHABET).strip()
    pred = invert_ohe(y_pred_err[i], ALPHABET).strip()
    if exp != pred:
        failures.append((exp, pred))

print(f"Total failures: {len(failures)}/500")
print(f"\n{'Expected':>10} {'Predicted':>10}")
for exp, pred in failures[:10]:
    print(f"{exp:>10} {pred:>10}")
# ─────────────────────────────────────────────────────────────────────────────

#### Written Answer – Task C4

**Q9.** What patterns did you observe in the failure cases? Is the model making systematic errors (e.g., always off by 1 in a specific digit position)?

*Failures tend to occur on larger sums (carry digits), and the model is often off by 1 in the units digit. This suggests it struggles with carry propagation — a known weakness of vanilla LSTMs on arithmetic tasks.*

---

---
## Part D (Bonus): Subtraction Seq2Seq

**Direction (Bonus — Optional):** Extend your seq2seq model to learn **subtraction** instead of (or in addition to) addition.

Changes required:
1. Modify `raw_sum_pairs` to generate `a - b` pairs (ensure `a >= b` to avoid negatives, or handle negative sign in the alphabet)
2. Update the alphabet to include `'-'` if needed
3. Re-train and evaluate

> For extra challenge: support **mixed operations** — randomly assign `+`,`*`,`/` or `-` per sample, and let the model figure out which operation to apply based on the string input.

In [ ]:
# BONUS TODO ──────────────────────────────────────────────────────────────────
# Implement subtraction (or mixed operations) seq2seq below.
# ─────────────────────────────────────────────────────────────────────────────

# Hint for subtraction:
def raw_subtract_pairs(n_examples, largest):
    X_list, y_list = [], []
    for _ in range(n_examples):
        a, b = sorted([randint(1, largest), randint(1, largest)], reverse=True)
        X_list.append([a, b])
        y_list.append(a - b)
    return X_list, y_list

print("Bonus task — implement subtraction seq2seq here!")

#### Written Answer – Part D

**Q10 (Bonus).** Did the model successfully learn subtraction? What was the hardest part of extending the approach?

*Your answer here (optional):*

---

---
## Final Summary

Fill in the table below after completing all parts:

| Task | Model | Metric | Your Result |
|------|-------|--------|-------------|
| Part A | Stacked LSTM | RMSE | |
| Part C1 | MLP Baseline | RMSE | |
| Part B | Seq2Seq LSTM | Exact-match Accuracy (%) | |
| Part C2 | Seq2Seq (scaled) | Exact-match Accuracy (%) | |
| Part D (Bonus) | Subtraction Seq2Seq | Exact-match Accuracy (%) | |

---

### Key Takeaways

Reflect briefly on the three most important things you learned from this assignment:

1. *Your answer here*
2. *Your answer here*
3. *Your answer here*

---

*Assignment based on: [MachineLearningMastery.com — Learn to Add Numbers with an Encoder-Decoder LSTM](https://machinelearningmastery.com/learn-add-numbers-seq2seq-recurrent-neural-networks/)*